In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import synergy_dataset as sd
from matplotlib.backends.backend_pdf import PdfPages

data_path = "./data/"

In [8]:
studies = pd.read_json("synergy_studies_validation.jsonl", lines=True)
studies_filtered = studies.sort_values("dataset_id").reset_index(drop=True)

recall_files = [
    "recalls_old1_nb.csv",
    #"recalls_old1_svm.csv",
    "recalls_new2_nb.csv",
    "recalls_new2_svm.csv",
    #"recalls_new2_mxbai_svm.csv",
    #"recalls_new2_e5_svm.csv",
]
recall_types = [
    #"ASR1.6 TF-IDF + NB",
    "ASR1.6",
    #"ASR1.6 TF-IDF + SVM",
    #"ASR2 TF-IDF + NB",
    #"ASR2 TF-IDF + SVM",
    "ASR2 u3",
    "ASR2 u4",
    #"ASR2 mxbai + SVM",
    #"ASR2 E5 + SVM",
]

In [9]:
def get_total_records(dataset_id):
    if dataset_id in {"Moran_2021_corrected", "Muthu_2021_corrected"}:
        return pd.read_csv(f"../src/datasets/{dataset_id}_shuffled_raw.csv").shape[0]
    else:
        return sd.Dataset(dataset_id).to_frame().shape[0]


def get_total_relevant(dataset_id):
    if dataset_id in {"Moran_2021_corrected", "Muthu_2021_corrected"}:
        return pd.read_csv(f"../src/datasets/{dataset_id}_shuffled_raw.csv")[
            "label_included"
        ].sum()
    else:
        return sd.Dataset(dataset_id).to_frame()["label_included"].sum()


total_relevant_dict = {
    dataset_id: get_total_relevant(dataset_id)
    for dataset_id in studies_filtered["dataset_id"].unique()
}

In [10]:
recall_dfs = [pd.read_csv(data_path + f) for f in recall_files]

# Add metadata to each DataFrame
for i, df in enumerate(recall_dfs):
    df["dataset_name"] = studies_filtered["dataset_id"].values
    df["Model"] = recall_types[i]
    df["prior_inclusions"] = studies_filtered["prior_inclusions"].apply(len)
    df["prior_exclusions"] = studies_filtered["prior_exclusions"].apply(len)
    df["simulation_id"] = df.groupby("dataset_name").cumcount() + 1

df_all = pd.concat(recall_dfs, ignore_index=True)

df_all_melted = df_all.melt(
    id_vars=[
        "dataset_name",
        "Model",
        "prior_inclusions",
        "prior_exclusions",
        "simulation_id",
    ],
    var_name="step",
    value_name="recall",
).dropna()

df_all_melted["step"] = df_all_melted["step"].astype(int)

df_all_melted["total_relevant"] = df_all_melted["dataset_name"].map(total_relevant_dict)

df_all_melted["relative_recall"] = df_all_melted["recall"] / (
    df_all_melted["total_relevant"] - df_all_melted["prior_inclusions"]
)

# Normalize step values to [0,1]
df_all_melted["relative_step"] = df_all_melted.groupby(["dataset_name", "Model"])[
    "step"
].transform(lambda x: x / x.max())

df_grouped = (
    df_all_melted.groupby(["dataset_name", "Model", "relative_step"])["relative_recall"]
    .mean()
    .reset_index()
)

df_grouped["Model"] = pd.Categorical(
    df_grouped["Model"], categories=recall_types, ordered=True
)
df_grouped.sort_values("Model", inplace=True)

In [11]:
def rank_loss(labels):
    """
    Custom loss that rewards 1s appearing earlier.
    labels: binary list or array, where higher value = better
    """
    labels = np.asarray(labels)
    Nx = len(labels)
    Ny = labels.sum()

    if Ny == 0 or Ny == Nx:
        return 0.0  # edge case: no 1s or all 1s

    loss = (Ny * (Nx - (Ny - 1) / 2) - np.cumsum(labels).sum()) / (Ny * (Nx - Ny))
    return loss


df_all_melted_srt = df_all_melted.sort_values(
    ["dataset_name", "Model", "simulation_id", "step"]
)

df_all_melted_srt["label"] = (
    df_all_melted_srt.groupby(["dataset_name", "Model", "simulation_id"])["recall"]
    .diff()
    .fillna(df_all_melted["recall"])  # First value is just recall itself
    .astype(int)
)

grouped_loss = (
    df_all_melted_srt.groupby(["dataset_name", "Model", "simulation_id"])["label"]
    .apply(rank_loss)
    .reset_index(name="loss")
)

# Compute standard deviation of loss per dataset and Model
std_loss_per_dataset_per_model = (
    grouped_loss.groupby(["dataset_name", "Model"])["loss"].std().reset_index()
)

# Compute mean std per Model (averaging across datasets)
std_loss_per_model = (
    std_loss_per_dataset_per_model.groupby(["Model"])["loss"].mean().reset_index()
)
std_loss_per_model[["Version", "Classifier"]] = std_loss_per_model["Model"].str.extract(
    r"(ASR\d+\.*\d*)\s(.*)"
)
pivot_std_df = std_loss_per_model.pivot(
    index="Classifier", columns="Version", values="loss"
)

# Compute mean of loss per dataset and Model
mean_loss_per_dataset_per_model = (
    grouped_loss.groupby(["dataset_name", "Model"])["loss"].mean().reset_index()
)

# Compute mean per Model (averaging across datasets)
mean_loss_per_model = (
    mean_loss_per_dataset_per_model.groupby(["Model"])["loss"].mean().reset_index()
)
mean_loss_per_model[["Version", "Classifier"]] = mean_loss_per_model[
    "Model"
].str.extract(r"(ASR\d+\.*\d*)\s(.*)")
pivot_df = mean_loss_per_model.pivot(
    index="Classifier", columns="Version", values="loss"
)

print("---MEAN---")
print(pivot_df)
print("\n----SD----")
print(pivot_std_df)

---MEAN---
Version          NaN      ASR2
Classifier                    
NaN         0.082131       NaN
u3               NaN  0.075746
u4               NaN  0.062264

----SD----
Version          NaN      ASR2
Classifier                    
NaN         0.008123       NaN
u3               NaN  0.007337
u4               NaN  0.004023


In [12]:
rows, cols = 5, 5
datasets = sorted(df_grouped["dataset_name"].unique())

with PdfPages("recall_comparison_old_new.pdf") as pdf:
    fig, axes = plt.subplots(rows, cols, figsize=(20, 20))
    axes = axes.flatten()

    for i, dataset in enumerate(datasets):
        ax = axes[i]
        df_subset = df_grouped[df_grouped["dataset_name"] == dataset]

        sns.lineplot(
            data=df_subset,
            x="relative_step",
            y="relative_recall",
            hue="Model",
            ax=ax,
            legend=True,
        )

        ax.set_xlabel("Proportion of Documents")
        ax.set_ylabel("Mean Recall")
        ax.set_title(f"{dataset}")

        max_relevant = (
            total_relevant_dict[dataset] - 1
        )  # Subtract at least 1 inclusion prior
        total_docs = (
            get_total_records(dataset) - 2
        )  # subtract at least 1 inclusion and 1 exclusion
        x_perfect = max_relevant / total_docs

        x_vals = np.linspace(0, 1, 10000)
        random_y = x_vals

        perfect_y = np.piecewise(
            x_vals,
            [x_vals <= x_perfect, x_vals > x_perfect],
            [lambda x: x / x_perfect, 1.0],
        )

        ax.plot(
            x_vals,
            random_y,
            color="gray",
            linestyle="--",
            linewidth=1,
            label="Random",
        )
        ax.plot(x_vals, perfect_y, color="gray", linewidth=1, label="Perfect")

    plt.suptitle("Mean Relative Recalls per SYNERGY Dataset")
    plt.tight_layout(rect=[0, 0.03, 1, 0.99])
    pdf.savefig(fig)
    plt.savefig("recall_comparison_old_new.png")
    plt.close(fig)